# Unity Model / Voxel Test Server

Run the code cell below in JupyterLab.

Test URLs:
- `http://127.0.0.1:8000/status` checks whether the server is running.
- `http://127.0.0.1:8000/model` downloads a local GLB if one exists in `local_models/`.
- Unity should send images to `http://127.0.0.1:8000/generate`.

For GLB mode, put `test_model.glb` or `sample.glb` inside `local_models/`.


In [4]:
from fastapi import FastAPI, UploadFile, File, Form, Request, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pathlib import Path
import shutil
import uuid
import threading
import uvicorn

# ------------------------------------------------------------
# Local paths
# ------------------------------------------------------------

BASE_DIR = Path.cwd()
IMAGE_DIR = BASE_DIR / "received_images"
MODEL_DIR = BASE_DIR / "local_models"

IMAGE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

MODEL_CANDIDATES = [
    MODEL_DIR / "test_model.glb",
    MODEL_DIR / "sample.glb",
]

# ------------------------------------------------------------
# FastAPI app
# ------------------------------------------------------------

app = FastAPI(title="Unity Model / Voxel Test Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ------------------------------------------------------------
# File helpers
# ------------------------------------------------------------

def save_uploaded_image(file: UploadFile) -> Path:
    """Save the image uploaded from Unity."""
    suffix = Path(file.filename).suffix or ".png"
    image_path = IMAGE_DIR / f"{uuid.uuid4()}{suffix}"

    with open(image_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    return image_path


def find_local_model():
    """Return the first local GLB test model that exists."""
    for model_path in MODEL_CANDIDATES:
        if model_path.exists():
            return model_path
    return None


def absolute_url(request: Request, route: str) -> str:
    """Build a URL that Unity can download."""
    return str(request.base_url).rstrip("/") + route

# ------------------------------------------------------------
# Voxel helpers
# ------------------------------------------------------------

def set_voxel(voxels: dict, x: int, y: int, z: int, kind: str, color: str):
    """Add one voxel inside a 64x64x64 grid."""
    if 0 <= x < 64 and 0 <= y < 64 and 0 <= z < 64:
        voxels[(x, y, z)] = {
            "x": x,
            "y": y,
            "z": z,
            "type": kind,
            "color": color,
        }


def add_box(voxels: dict, x0: int, x1: int, y0: int, y1: int, z0: int, z1: int, kind: str, color: str):
    """Add a filled box of adjacent voxels."""
    for x in range(x0, x1 + 1):
        for y in range(y0, y1 + 1):
            for z in range(z0, z1 + 1):
                set_voxel(voxels, x, y, z, kind, color)


def add_wall_box(voxels: dict, x0: int, x1: int, y0: int, y1: int, z0: int, z1: int, kind: str, color: str):
    """Add only the outer surface of a rectangular building volume."""
    for y in range(y0, y1 + 1):
        for x in range(x0, x1 + 1):
            set_voxel(voxels, x, y, z0, kind, color)
            set_voxel(voxels, x, y, z1, kind, color)
        for z in range(z0, z1 + 1):
            set_voxel(voxels, x0, y, z, kind, color)
            set_voxel(voxels, x1, y, z, kind, color)


def add_post(voxels: dict, x: int, z: int, y0: int, y1: int, color: str):
    """Add one vertical wooden post."""
    for y in range(y0, y1 + 1):
        set_voxel(voxels, x, y, z, "wood", color)


def generate_test_house_voxels_64(max_voxels: int = 12000):
    """
    Generate dense Minecraft-style test voxels in a 64x64x64 grid.

    This is a placeholder for a future AI model. It does not use the uploaded image yet.

    Important display rule:
    - Server coordinates are adjacent integers.
    - Unity should place each cube with spacing = voxel_size.
    - Unity should scale each cube to voxel_size.
    - Neighboring voxels then touch face-to-face like Minecraft blocks.
    """
    voxels = {}

    wall = "#E8DDC7"
    wood = "#A66A3D"
    dark_wood = "#6B3F24"
    roof = "#5F6264"
    roof_dark = "#3D3F40"
    foundation = "#B8B8B8"
    window = "#F4F6F7"
    stone = "#9A9A8A"
    plant = "#4D7C3A"

    # Foundation and floor.
    add_box(voxels, 10, 54, 0, 0, 12, 48, "foundation", foundation)
    add_box(voxels, 14, 50, 1, 1, 18, 42, "wood_floor", dark_wood)

    # Main house walls.
    add_wall_box(voxels, 16, 48, 2, 16, 18, 42, "wall", wall)

    # Wooden posts and beams.
    for x, z in [(16, 18), (32, 18), (48, 18), (16, 42), (32, 42), (48, 42), (16, 30), (48, 30)]:
        add_post(voxels, x, z, 1, 20, wood)

    add_box(voxels, 16, 48, 17, 17, 18, 18, "wood", wood)
    add_box(voxels, 16, 48, 17, 17, 42, 42, "wood", wood)
    add_box(voxels, 16, 16, 17, 17, 18, 42, "wood", wood)
    add_box(voxels, 48, 48, 17, 17, 18, 42, "wood", wood)

    # Door and shoji-style windows.
    add_box(voxels, 20, 26, 3, 13, 17, 17, "door", "#D8D2C2")
    add_box(voxels, 30, 44, 6, 14, 17, 17, "window", window)
    add_box(voxels, 49, 49, 6, 14, 26, 38, "window", window)

    # Window grid frames.
    for x in range(30, 45, 4):
        add_box(voxels, x, x, 6, 14, 16, 16, "wood", wood)
    for y in range(6, 15, 4):
        add_box(voxels, 30, 44, y, y, 16, 16, "wood", wood)
    for z in range(26, 39, 4):
        add_box(voxels, 50, 50, 6, 14, z, z, "wood", wood)

    # Dense stepped roof. The extra lower layer removes visible diagonal gaps.
    center_z = 30
    for x in range(8, 57):
        for z in range(8, 53):
            slope = abs(z - center_z)
            y = 29 - slope // 2
            if y >= 18:
                color = roof_dark if x % 6 == 0 or z % 6 == 0 else roof
                set_voxel(voxels, x, y, z, "roof", color)
                set_voxel(voxels, x, y - 1, z, "roof", color)

    add_box(voxels, 8, 56, 30, 30, 30, 30, "roof_ridge", roof_dark)
    add_box(voxels, 8, 56, 18, 18, 8, 8, "roof_edge", roof_dark)
    add_box(voxels, 8, 56, 18, 18, 52, 52, "roof_edge", roof_dark)

    # Small entrance roof and posts.
    for x in range(10, 30):
        for z in range(8, 17):
            y = 15 + (z - 8) // 4
            set_voxel(voxels, x, y, z, "small_roof", roof)
            set_voxel(voxels, x, y - 1, z, "small_roof", roof)

    add_post(voxels, 10, 10, 1, 15, wood)
    add_post(voxels, 28, 10, 1, 15, wood)

    # Garden stones and plants.
    for x, z in [(12, 6), (18, 6), (24, 7), (30, 7), (42, 52), (48, 54), (54, 55)]:
        add_box(voxels, x, x + 2, 1, 1, z, z + 1, "stone", stone)

    for x, z in [(8, 48), (10, 50), (56, 18), (58, 20)]:
        add_box(voxels, x, x + 1, 1, 4, z, z + 1, "plant", plant)

    # Do not stride-sample the list. Striding creates checkerboard gaps.
    voxel_list = list(voxels.values())
    return voxel_list[:max_voxels]

# ------------------------------------------------------------
# API endpoints
# ------------------------------------------------------------

@app.get("/status")
def status():
    """Check whether the server is running."""
    return {
        "status": "ok",
        "message": "Unity model / voxel test server is running.",
        "model_available": find_local_model() is not None,
    }


@app.get("/health")
def health():
    """Alias for /status. Kept for compatibility with older tests."""
    return status()


@app.get("/model")
def get_model():
    """Send a local GLB file to Unity."""
    model_path = find_local_model()
    if model_path is None:
        raise HTTPException(
            status_code=404,
            detail="No GLB found. Put test_model.glb or sample.glb inside local_models/.",
        )

    return FileResponse(
        path=model_path,
        media_type="model/gltf-binary",
        filename=model_path.name,
    )


@app.post("/generate")
async def generate(
    request: Request,
    file: UploadFile = File(...),
    response_mode: str = Form("voxels"),
    max_voxels: int = Form(12000),
):
    """
    Receive one image from Unity and return test output.

    response_mode:
        "model"  -> return a local GLB model URL
        "voxels" -> return 64x64x64 voxel coordinates
        "both"   -> return both model URL and voxel coordinates
    """
    response_mode = response_mode.lower().strip()
    if response_mode not in {"model", "voxels", "both"}:
        raise HTTPException(status_code=400, detail="response_mode must be model, voxels, or both")

    max_voxels = max(1, int(max_voxels))
    image_path = save_uploaded_image(file)
    print(f"Received image from Unity: {image_path}")

    result = {
        "status": "done",
        "response_mode": response_mode,
    }

    if response_mode in {"model", "both"}:
        if find_local_model() is None:
            result["model_error"] = "No local GLB found in local_models/."
        else:
            result["model_url"] = absolute_url(request, "/model")

    if response_mode in {"voxels", "both"}:
        voxels = generate_test_house_voxels_64(max_voxels=max_voxels)
        result["grid_size"] = 64
        result["voxel_count"] = len(voxels)
        result["voxels"] = voxels

    return result

# ------------------------------------------------------------
# JupyterLab server controls
# ------------------------------------------------------------

try:
    server
except NameError:
    server = None

try:
    server_thread
except NameError:
    server_thread = None


def start_server(host="127.0.0.1", port=8000):
    """Start the FastAPI server in a background thread for JupyterLab."""
    global server, server_thread

    if server_thread is not None and server_thread.is_alive():
        print(f"Server is already running at http://{host}:{port}")
        print("If you changed the code, run stop_server(), wait one second, then run start_server() again.")
        return

    config = uvicorn.Config(app, host=host, port=port, log_level="info")
    server = uvicorn.Server(config)
    server_thread = threading.Thread(target=server.run, daemon=True)
    server_thread.start()
    print(f"Server started at http://{host}:{port}")


def stop_server():
    """Stop the background server."""
    global server

    if server is not None:
        server.should_exit = True
        print("Server stopping...")
    else:
        print("Server is not running.")

start_server()


Server is already running at http://127.0.0.1:8000
If you changed the code, run stop_server(), wait one second, then run start_server() again.
